# TP — Tokenisation & exploration de jeux de données textuelles (NLTK & spaCy)

**Durée** : 2–3h &nbsp;&nbsp;|&nbsp;&nbsp; **Langue** : Français &nbsp;&nbsp;|&nbsp;&nbsp; **Niveau** : L3 / M1

## Objectifs pédagogiques
À la fin du TP, vous saurez :
- Charger et **explorer** un corpus textuel (CSV).
- Appliquer un **prétraitement** : nettoyage, tokenisation, stopwords, normalisation.
- Mesurer la **richesse lexicale** (TTR, hapax) et produire des **n-grammes** (bigrams).
- Exploiter **spaCy** pour les **lemmes**, **POS**, **entités nommées** et **dépendances**.
- Implémenter un **analyseur simple** Sujet–Verbe–Objet (S‑V‑O).
- **Visualiser** des fréquences et **exporter** des résultats pour un rapport.

## Consignes & livrables
- Travail **individuel** ou **binôme** (selon consignes de l’enseignant).
- **Ne modifiez pas** l’ordre des cellules ; **complétez les TODO**.
- Générer **au moins 2 graphiques** (fréquences, POS, étiquettes NER…).
- Exporter au moins **4 fichiers CSV** (voir section Export).
- Rédiger une **synthèse** (10–15 lignes) répondant aux questions d’exploration.

**À rendre** : le notebook complété (`.ipynb`) + CSV générés.

## 0) Pré-requis & installation (exécuter si nécessaire)
- Python 3.x, Jupyter, `pandas`, `matplotlib`, `nltk`, `spacy`
- Modèles spaCy FR : `fr_core_news_md` (ou `fr_core_news_sm`)

> Si vous êtes hors-ligne, assurez-vous que les paquets/modèles sont déjà installés.

In [ ]:
# Décommentez au besoin :
# !pip install -U spacy nltk pandas matplotlib scikit-learn
# !python -m spacy download fr_core_news_md  # ou: fr_core_news_sm

import nltk
# nltk.download('punkt')
# nltk.download('stopwords')

import warnings; warnings.filterwarnings("ignore")

## 1) Charger le corpus
Par défaut, le notebook attend `mini_corpus_fr.csv` (colonne `text`).  
Vous pouvez **remplacer** par votre propre corpus (même format).

### TODO
1. Charger le CSV dans un DataFrame `df` (afficher `head()`).
2. Vérifier : nombre de lignes, valeurs manquantes, doublons.
3. Ajouter une colonne `n_chars` (longueur des textes) et afficher stats descriptives.

In [ ]:
import os, pandas as pd

CSV_PATH = "mini_corpus_fr.csv"  # ⚠️ modifiez si vous utilisez votre propre corpus

if not os.path.exists(CSV_PATH):
    print("⚠️ Fichier introuvable. Placez un CSV avec une colonne 'text' à côté du notebook.")
    
# === TODO 1 : charger le CSV dans df et afficher head() ===
df = pd.read_csv(CSV_PATH)
display(df.head())

# === TODO 2 : vérifications ===
n_rows = len(df)
n_missing = df['text'].isna().sum() if 'text' in df else None
n_dups = df.duplicated(subset=['text']).sum() if 'text' in df else None
print(f"Lignes: {n_rows} | NaN dans 'text': {n_missing} | Doublons (text): {n_dups}")

# === TODO 3 : longueur des textes ===
df['n_chars'] = df['text'].astype(str).str.len()
display(df['n_chars'].describe())

## 2) Prétraitement + Tokenisation (NLTK)
Nous allons normaliser les textes et **tokeniser** avec NLTK.

### TODO
1. Implémenter `simple_clean(text)` : suppression URLs/emails, espaces multiples…
2. Implémenter `nltk_tokenize(text)` : tokenisation FR + retrait de la ponctuation.
3. Créer `tokens_norm` (minuscules, alphabetiques), puis `tokens_nostop` (sans stopwords).
4. Calculer **mesures lexicales** par document : nb tokens, vocab, **TTR**, **hapax_ratio**.

In [ ]:
import re, string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import numpy as np

# Stopwords FR (si non téléchargés, la liste sera vide)
try:
    SW_FR = set(stopwords.words("french"))
except LookupError:
    SW_FR = set()
    print("⚠️ Stopwords FR non disponibles. Lancez nltk.download('stopwords').")

PUNCT = set(string.punctuation + "“”«»…’‹›–—")

def simple_clean(text: str) -> str:
    """TODO: nettoyer le texte (URLs, emails, espaces)."""
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def nltk_tokenize(text: str):
    """TODO: tokeniser en FR et retirer la ponctuation."""
    toks = word_tokenize(text, language="french")
    toks = [t for t in toks if t not in PUNCT]
    return toks

def normalize_tokens(tokens):
    return [t.lower() for t in tokens if t.isalpha()]

def filter_stop(tokens):
    return [t for t in tokens if t not in SW_FR]

def lexical_measures(tokens):
    if not tokens:
        return dict(tokens=0, vocab=0, ttr=0.0, hapax_ratio=0.0)
    vocab = set(tokens)
    counts = Counter(tokens)
    hapax = sum(1 for w,c in counts.items() if c==1)
    return {
        "tokens": len(tokens),
        "vocab": len(vocab),
        "ttr": len(vocab)/len(tokens),
        "hapax_ratio": hapax/len(tokens)
    }

docs = df['text'].astype(str).apply(simple_clean).tolist()

all_tokens, per_doc_stats = [], []
per_doc_tokens = []   # conserver les tokens nettoyés par doc

for i, txt in enumerate(docs, start=1):
    toks = nltk_tokenize(txt)
    toks_norm = normalize_tokens(toks)
    toks_nostop = filter_stop(toks_norm)
    per_doc_tokens.append(toks_nostop)
    all_tokens.extend(toks_nostop)
    per_doc_stats.append({ "doc_id": i, **lexical_measures(toks_nostop) })

per_doc_stats_df = pd.DataFrame(per_doc_stats)
global_measures = lexical_measures(all_tokens)

display(per_doc_stats_df.head())
print("Mesures globales :", global_measures)

### Questions d’exploration (Q1–Q4)
- **Q1.** Comparez la **TTR** et le **hapax_ratio** entre documents : lesquels sont les plus « riches » lexicalement ? Hypothèses ?
- **Q2.** Quels sont les **20 tokens** les plus fréquents (hors stopwords) ? Sont-ils informatifs ?
- **Q3.** Comment l’ajout/retrait de certains **stopwords** influencerait vos résultats ?
- **Q4.** Proposez **2 règles** supplémentaires de nettoyage adaptées à votre corpus (exemples : nombres, emojis, tokens très courts).

> Répondez directement ci-dessous.

**Réponses (Q1–Q4)**  
_Rédigez votre analyse ici…_

## 3) N-grammes
Nous allons extraire des **bigrams** (option : trigrams) sur les tokens nettoyés.

### TODO
1. Implémenter `bigrams(tokens)`.
2. Calculer le **Top 20** des bigrams.
3. Proposer un **filtre** pour supprimer les bigrams peu informatifs (ex. contenant stopwords).

In [ ]:
from collections import Counter

def bigrams(tokens):
    return list(zip(tokens, tokens[1:]))

all_bigrams = []
for toks in per_doc_tokens:
    all_bigrams.extend(bigrams(toks))

big_counts = Counter(all_bigrams).most_common(20)
big_df = pd.DataFrame([{'bigram': ' '.join(b), 'freq': f} for (b,f) in big_counts])
display(big_df)

## 4) Visualisation (matplotlib)
Tracez un **histogramme** des 20 tokens les plus fréquents.

### TODO
- Construire la liste `(mots, fréquences)` à partir de `all_tokens`.
- Tracer une **barre** simple (sans couleurs spécifiques).

> Note : Une figure par graphique (pas de sous‑plots).

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

top20 = Counter(all_tokens).most_common(20)
mots, freqs = zip(*top20) if top20 else ([], [])

plt.figure(figsize=(10,5))
plt.bar(mots, freqs)
plt.xticks(rotation=45, ha='right')
plt.title("Top 20 tokens (hors stopwords)")
plt.xlabel("Token"); plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()

## 5) Annotation spaCy : lemmes, POS, NER
Nous utiliserons `fr_core_news_md` (ou `sm`) pour enrichir le corpus.

### TODO
1. Charger `nlp` (FR). Si le modèle n’est pas dispo, afficher un message clair.
2. Pour chaque document : construire `tokens_df` (text, lemma, pos, dep, is_stop, is_alpha).
3. Extraire `ents_df` (entités nommées) et `pos_df` (distribution POS par doc).
4. Afficher le **Top POS** global et **Top étiquettes NER**.

In [ ]:
import spacy

def load_fr():
    try:
        return spacy.load("fr_core_news_md")
    except Exception:
        try:
            return spacy.load("fr_core_news_sm")
        except Exception as e:
            print("❌ Aucun modèle FR n'est dispo. Exécutez :")
            print("   python -m spacy download fr_core_news_md  (ou fr_core_news_sm)")
            raise e

nlp = load_fr()

token_rows, ent_rows, pos_rows = [], [], []

for i, txt in enumerate(docs, start=1):
    doc = nlp(txt)
    for t in doc:
        if not t.is_space:
            token_rows.append({
                "doc_id": i, "text": t.text, "lemma": t.lemma_,
                "pos": t.pos_, "dep": t.dep_, "is_stop": t.is_stop, "is_alpha": t.is_alpha
            })
    for e in doc.ents:
        ent_rows.append({"doc_id": i, "ent_text": e.text, "label": e.label_})
    # comptage POS par doc
    counts = {}
    for t in doc:
        if not t.is_space:
            counts[t.pos_] = counts.get(t.pos_, 0) + 1
    for pos, c in counts.items():
        pos_rows.append({"doc_id": i, "pos": pos, "count": c})

import pandas as pd
tokens_df = pd.DataFrame(token_rows)
ents_df = pd.DataFrame(ent_rows)
pos_df = pd.DataFrame(pos_rows)

display(tokens_df.head())
display(ents_df['label'].value_counts().head(10))
display(pos_df.groupby('pos')['count'].sum().sort_values(ascending=False).head(10))

### Questions d’exploration (Q5–Q8)
- **Q5.** Quels **lemmes** dominent ? Expliquez la différence entre **forme** et **lemme** sur un exemple.
- **Q6.** Interprétez la distribution des **POS** (catégories grammaticales). Est‑elle cohérente avec le type de documents ?
- **Q7.** Quelles **entités nommées** (labels NER) ressortent ? Sont‑elles correctes ? Donnez 2 cas d’erreurs possibles.
- **Q8.** Proposez un **filtrage** (par POS/lemme) pour améliorer votre top tokens.

## 6) Mini‑analyseur S‑V‑O (dépendances spaCy)
Heuristique simple basée sur le verbe **ROOT** et ses sujets/objets.

### TODO
1. Implémenter `extract_svos(doc)` : récupérer les triplets (sujet, verbe_lemme, objet/complément).
2. Appliquer sur tout le corpus → `svo_df`.
3. **Analyser** 3–5 triplets : sont‑ils pertinents ? Cas d’erreurs ?

In [ ]:
def extract_svos(doc):
    svos = []
    for v in doc:
        if v.pos_ == "VERB" and v.dep_ == "ROOT":
            subs = [c for c in v.lefts if c.dep_.startswith("nsubj")]
            objs = [c for c in v.rights if c.dep_ in ("obj", "iobj", "obl")]
            for s in subs:
                for o in objs:
                    svos.append((s.text, v.lemma_, o.text))
    return svos

svo_rows = []
for i, txt in enumerate(docs, start=1):
    doc = nlp(txt)
    for s, v, o in extract_svos(doc):
        svo_rows.append({"doc_id": i, "subject": s, "verb_lemma": v, "object_or_obl": o})

svo_df = pd.DataFrame(svo_rows)
display(svo_df.head(10))

## 7) Visualisations complémentaires
Créez **au moins 1** graphique supplémentaire parmi :
- Top 10 **lemmes** (hors stopwords, hors mots très courts).
- Distribution **POS** globale (barres).
- Top **étiquettes NER**.

> Utilisez `matplotlib` (pas de couleurs spécifiques).

In [ ]:
# Exemple : distribution POS globale
import matplotlib.pyplot as plt

pos_global = pos_df.groupby('pos')['count'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,4))
plt.bar(pos_global.index, pos_global.values)
plt.title("Distribution POS globale")
plt.xlabel("POS"); plt.ylabel("Fréquence")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8) Export des résultats (CSV)

### TODO
Exporter au **format CSV** (dans le dossier du notebook) :
- `per_doc_stats.csv` : mesures lexicales par document
- `top_tokens.csv` : top tokens global
- `bigrams.csv` : top bigrams global
- `tokens_spacy.csv` : tokens avec lemmes/POS/dépendances
- `entities_spacy.csv` : entités nommées
- `pos_counts.csv` : distribution POS par document
- `svo.csv` : triplets S‑V‑O

In [ ]:
from collections import Counter

per_doc_stats_df.to_csv("per_doc_stats.csv", index=False, encoding="utf-8")

from collections import Counter
top_df = pd.DataFrame(Counter([t for t in all_tokens]).most_common(), columns=["token","freq"])
top_df.to_csv("top_tokens.csv", index=False, encoding="utf-8")

# bigrams
big_df = pd.DataFrame([{'bigram':' '.join(b), 'freq':f} for (b,f) in Counter(all_bigrams).most_common()], columns=['bigram','freq'])
big_df.to_csv("bigrams.csv", index=False, encoding="utf-8")

# spaCy exports
tokens_df.to_csv("tokens_spacy.csv", index=False, encoding="utf-8")
ents_df.to_csv("entities_spacy.csv", index=False, encoding="utf-8")
pos_df.to_csv("pos_counts.csv", index=False, encoding="utf-8")

# SVO
try:
    svo_df.to_csv("svo.csv", index=False, encoding="utf-8")
except NameError:
    print("SVO non disponible (spaCy requis).")
    
print("✅ Exports écrits dans le dossier courant.")

## 9) (Bonus) Représentations vectorielles
Avec `scikit-learn`, calculez une **matrice TF‑IDF** sur `df['text']` prétraité.

### TODO (optionnel)
1. Construire une liste de documents prétraités (chaîne de tokens).
2. Utiliser `TfidfVectorizer` (ngram_range, min_df, max_df à tester).
3. Afficher les **top termes** par poids pour 2–3 documents.

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# docs_joined = [" ".join(toks) for toks in per_doc_tokens]
# vec = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_df=0.95)
# X = vec.fit_transform(docs_joined)
# feature_names = vec.get_feature_names_out()
# # Exemple : top termes du 1er doc
# row = X[0].toarray().ravel()
# top_idx = row.argsort()[-10:][::-1]
# [(feature_names[i], round(row[i],3)) for i in top_idx]

## 10) Synthèse (à rédiger)
Répondez brièvement aux points suivants (10–15 lignes) :
- Qu’avez‑vous observé sur la **richesse lexicale** et les **top tokens** ?
- Les **n-grammes** apportent‑ils des informations supplémentaires ?
- Quels **POS** et **NER** dominent et pourquoi ?
- Évaluez la qualité du **mini‑analyseur S‑V‑O** (forces/faiblesses). Idées d’amélioration ?